In [36]:
import sys
import os
from pyltp import Segmentor, Postagger, Parser, SementicRoleLabeller
from nltk import DependencyGraph

class LtpLanguageAnalysis(object):
    def __init__(self, model_dir="/Users/sunhongchao/Documents/craft/09_Dialogue/resources/ltp_data_v3.4.0"):
        self.segmentor = Segmentor()
        self.segmentor.load(os.path.join(model_dir, "cws.model"))
        self.postagger = Postagger()
        self.postagger.load(os.path.join(model_dir, "pos.model"))
        self.parser = Parser()
        self.parser.load(os.path.join(model_dir, "parser.model"))
        self.labeler = SementicRoleLabeller()
        self.labeler.load(os.path.join(model_dir, "pisrl.model"))

    def analyze(self, text):
        # 分词
        words = self.segmentor.segment(text)
        # print('\t'.join(words))
             
        # 词性标注
        postags = self.postagger.postag(words)
        # print('\t'.join(postags))

        # 依存句法分析

        arcs = self.parser.parse(words, postags)
        # rely_id = [arc.head for arc in arcs]  # 提取依存父节点id
        # relation = [arc.relation for arc in arcs]  # 提取依存关系
        # heads = ['Root' if id == 0 else words[id - 1] for id in rely_id]  # 匹配依存父节点词语

        # par_result = ''
        # for i in range(len(words)):
        #     if arcs[i].head == 0:
        #         arcs[i].relation = "ROOT"
        #     par_result += "\t" + words[i] + "(" + arcs[i].relation + ")" + "\t" + postags[i] + "\t" + str(arcs[i].head) + "\t" + arcs[i].relation + "\n"
        # # print(par_result)
        # conlltree = DependencyGraph(par_result)  # 转换为依存句法图
        # tree = conlltree.tree()  # 构建树结构
        # tree.draw()  # 显示输出的树

        roles = self.labeler.label(words, postags, arcs)  # 语义角色标注
        tmp_words = list(words)
        # print('\n\n')
        #print('tmp words', tmp_words)
        # print('\n\n')
        for role in roles:
            print(words[role.index], end=' ')
            print(role.index, "".join(["%s:(%d,%d)" % (arg.name, arg.range.start, arg.range.end) for arg in role.arguments]))
            for arg in role.arguments:
                if arg.name == 'A1':
                    a0_str_before = ''.join(tmp_words[0:arg.range.start]) 
                    a0_str_after = ''.join(tmp_words[arg.range.end+1::]) 
                    #a0_str = ''.join(words[arg.range.start:arg.range.end])
                    a0_str = '###谁###'
                    #print(a0_str_before + a0_str + a0_str_after)
                    print( a0_str_before + a0_str + a0_str_after + '?')
                if arg.name == 'TMP':
                    a0_str_before = ''.join(tmp_words[0:arg.range.start]) 
                    a0_str_after = ''.join(tmp_words[arg.range.end+1::]) 
                    #a0_str = ''.join(words[arg.range.start:arg.range.end])
                    a0_str = '###什么时间###'
                    print( a0_str_before + a0_str +  a0_str_after + '?')  

    def release_model(self):
        # 释放模型
        self.segmentor.release()
        self.postagger.release()
        self.parser.release()
        self.labeler.release()


if __name__ == '__main__':
    ltp = LtpLanguageAnalysis()
    ltp.analyze("外商投资企业应于每年1月1日至6月30日通过国家企业信用信息公示系统提交上一年度的年度报告")
    ltp.release_model()

提交 18 A1:(0,2)TMP:(4,10)MNR:(11,17)A1:(19,24)
###谁###应于每年1月1日至6月30日通过国家企业信用信息公示系统提交上一年度的年度报告?
外商投资企业应###什么时间###通过国家企业信用信息公示系统提交上一年度的年度报告?
外商投资企业应于每年1月1日至6月30日通过国家企业信用信息公示系统提交###谁###?
